In [3]:
import albumentations as A
import cv2
import os
import glob
from tqdm import tqdm
from ultralytics import YOLO
import torch

/home/var-roman/anaconda3/envs/Analysis_and_processing/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
base_dir = 'nike_dataset/train'
images_dir = os.path.join(base_dir, 'images')
labels_dir = os.path.join(base_dir, 'labels')

transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=15, p=0.5),
    A.RandomScale(scale_limit=0.1, p=0.5),
    A.Perspective(p=0.2),
    A.RandomBrightnessContrast(p=0.5),
    A.GaussianBlur(blur_limit=3, p=0.2),
    A.GaussNoise(p=0.2),
    A.HueSaturationValue(p=0.3),
    A.CoarseDropout(
        num_holes_range=(1, 8),
        hole_height_range=(8, 30),
        hole_width_range=(8, 30),
        p=0.2
    )
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))

def read_label(label_path):
    bboxes, classes = [], []
    if os.path.exists(label_path):
        with open(label_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    classes.append(int(parts[0]))
                    bboxes.append([float(x) for x in parts[1:]])
    return bboxes, classes

def save_label(save_path, bboxes, classes):
    with open(save_path, 'w') as f:
        for cls, bbox in zip(classes, bboxes):
            bbox = [min(max(x, 0.0), 1.0) for x in bbox]
            f.write(f"{cls} {' '.join(map(str, bbox))}\n")

image_paths = glob.glob(os.path.join(images_dir, '*.jpg'))
print(f"Всього зображень: {len(image_paths)}")

LIMIT = 500
count = 0

for img_path in tqdm(image_paths, desc="Аугментація"):
    if count >= LIMIT:
        break

    if "_aug_" in img_path: continue

    image = cv2.imread(img_path)
    if image is None: continue
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    base_name = os.path.basename(img_path).rsplit('.', 1)[0]
    label_path = os.path.join(labels_dir, base_name + '.txt')

    bboxes, class_labels = read_label(label_path)

    if not bboxes: continue

    for i in range(1):
        try:
            augmented = transform(image=image, bboxes=bboxes, class_labels=class_labels)

            if len(augmented['bboxes']) > 0:
                new_filename = f"{base_name}_aug_{i}"

                out_img = cv2.cvtColor(augmented['image'], cv2.COLOR_RGB2BGR)
                cv2.imwrite(os.path.join(images_dir, new_filename + '.jpg'), out_img)
                save_label(os.path.join(labels_dir, new_filename + '.txt'),
                           augmented['bboxes'],
                           augmented['class_labels'])
        except Exception:
            pass

    count += 1

print(f'Додано {count} нових зображень')

Всього зображень: 4279


Аугментація:  13%|█▎        | 563/4279 [00:03<00:26, 141.67it/s]

Додано 500 нових зображень


In [5]:
if torch.cuda.is_available():
    torch.cuda.empty_cache()

def train_nike():
    yaml_path = os.path.abspath('nike_dataset/data.yaml')

    model = YOLO('yolo11n.pt')
    results = model.train(
        data=yaml_path,
        epochs=3,
        imgsz=640,
        batch=32,
        device=0,
        name='nike_mk',
        patience=1,
        workers=0)

    print("Навчання завершено")

train_nike()

Ultralytics 8.3.235 🚀 Python-3.11.14 torch-2.4.0 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 5762MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/var-roman/Desktop/For_studying/7_semester/PR_PZ/MCW/nike_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=3, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=nike_mk3, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto

In [6]:
video_path = "nike_dataset/nike_video.webm"
model_path = "runs/detect/nike_mk3/weights/best.pt"

def process_video():
    model = YOLO(model_path)

    results = model.predict(
        source=video_path,
        save=True,
        conf=0.5,
        device=0,
        stream=True,
        project="runs/detect",
        name="video_result_nike")

    for result in results:
        pass
    print("Відео готове")

process_video()


video 1/1 (frame 1/1093) /home/var-roman/Desktop/For_studying/7_semester/PR_PZ/MCW/nike_dataset/nike_video.webm: 384x640 (no detections), 10.8ms
video 1/1 (frame 2/1093) /home/var-roman/Desktop/For_studying/7_semester/PR_PZ/MCW/nike_dataset/nike_video.webm: 384x640 (no detections), 8.5ms
video 1/1 (frame 3/1093) /home/var-roman/Desktop/For_studying/7_semester/PR_PZ/MCW/nike_dataset/nike_video.webm: 384x640 (no detections), 6.6ms
video 1/1 (frame 4/1093) /home/var-roman/Desktop/For_studying/7_semester/PR_PZ/MCW/nike_dataset/nike_video.webm: 384x640 (no detections), 7.8ms
video 1/1 (frame 5/1093) /home/var-roman/Desktop/For_studying/7_semester/PR_PZ/MCW/nike_dataset/nike_video.webm: 384x640 (no detections), 6.3ms
video 1/1 (frame 6/1093) /home/var-roman/Desktop/For_studying/7_semester/PR_PZ/MCW/nike_dataset/nike_video.webm: 384x640 1 nike, 7.1ms
video 1/1 (frame 7/1093) /home/var-roman/Desktop/For_studying/7_semester/PR_PZ/MCW/nike_dataset/nike_video.webm: 384x640 1 nike, 6.4ms
video 1/